# Praktikum Data Warehouse SIM-PETANI SUMUT
ETL PostgreSQL Neon menggunakan Google Colab.

Tujuan:
1. Menghubungkan DB_PUSAT_SUMUT dan DW_SUMUT.
2. Membuat star schema.
3. Mengekstrak, membersihkan, dan memuat data petani.
4. Membuat snapshot harian yang aman dijalankan ulang.
5. Membuat SQL Views dan memvalidasi hasil.

Arsitektur: DB_KARO + DB_DELI_SERDANG + DB_LANGKAT → DB_PUSAT_SUMUT → DW_SUMUT → Dashboard.

Notebook membaca `public.petani_pusat` dari DB_PUSAT_SUMUT. Pastikan sinkronisasi dari database kabupaten ke pusat telah dilakukan. Jangan membagikan connection string atau password.


## 0. Persiapan
Buka Google Colab, buat notebook baru, lalu jalankan sel secara berurutan. Siapkan dua connection string Neon: DB_PUSAT_SUMUT dan DW_SUMUT. Gunakan SSL dan role database dengan hak akses minimum.


In [ ]:
!pip -q install "psycopg2-binary>=2.9,<3" pandas

## 1. Import library dan masukkan kredensial

In [ ]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from getpass import getpass
from decimal import Decimal, InvalidOperation
from datetime import datetime
from zoneinfo import ZoneInfo

DB_PUSAT = getpass("Masukkan connection string DB_PUSAT_SUMUT: ")
DB_WAREHOUSE = getpass("Masukkan connection string DW_SUMUT: ")

if not DB_PUSAT.strip() or not DB_WAREHOUSE.strip():
    raise ValueError("Kedua connection string wajib diisi.")

print("Kredensial dimasukkan. Jangan tampilkan atau bagikan nilainya.")


## 2. Uji koneksi

In [ ]:
def test_connection(connection_string, label):
    conn = None
    try:
        # Hindari startup options statement_timeout pada Neon pooled endpoint.
        conn = psycopg2.connect(connection_string, connect_timeout=15)
        with conn.cursor() as cur:
            cur.execute("SELECT current_database(), current_user;")
            db_name, db_user = cur.fetchone()
        print(f"BERHASIL: {label} | Database: {db_name} | User: {db_user}")
        return True
    except Exception as exc:
        print(f"GAGAL: {label} | {type(exc).__name__}: {exc}")
        return False
    finally:
        if conn:
            conn.close()

ok_pusat = test_connection(DB_PUSAT, "DB_PUSAT_SUMUT")
ok_dw = test_connection(DB_WAREHOUSE, "DW_SUMUT")
if not (ok_pusat and ok_dw):
    raise RuntimeError("Periksa koneksi sebelum melanjutkan.")


## 3. Periksa struktur dan jumlah data sumber

In [ ]:
conn = psycopg2.connect(DB_PUSAT, connect_timeout=15)
try:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT column_name, data_type, is_nullable
            FROM information_schema.columns
            WHERE table_schema='public' AND table_name='petani_pusat'
            ORDER BY ordinal_position;
        """)
        schema_rows = cur.fetchall()
        for row in schema_rows:
            print(row)

        required = {"kode_petani", "nama_petani", "desa", "komoditas",
                    "luas_lahan", "wilayah_sumber", "updated_at"}
        available = {row[0] for row in schema_rows}
        missing = required - available
        if missing:
            raise ValueError(f"Kolom sumber belum tersedia: {sorted(missing)}")

        cur.execute("SELECT COUNT(*) FROM public.petani_pusat;")
        print("TOTAL DATA SUMBER:", cur.fetchone()[0])

        cur.execute("""
            SELECT wilayah_sumber, COUNT(*)
            FROM public.petani_pusat
            GROUP BY wilayah_sumber ORDER BY wilayah_sumber;
        """)
        print("DATA PER WILAYAH:")
        for row in cur.fetchall():
            print(row)
finally:
    conn.close()


## 4. Buat skema Data Warehouse
Skema menggunakan tiga tabel dimensi dan satu tabel fakta. Tabel fakta menyimpan snapshot harian. Kolom nama_petani disertakan untuk kebutuhan laporan internal; jangan tampilkan identitas pribadi di dashboard publik.


In [ ]:
DDL = """
CREATE SCHEMA IF NOT EXISTS dw;

CREATE TABLE IF NOT EXISTS dw.dim_wilayah (
    id_wilayah BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    kabupaten VARCHAR(100) NOT NULL,
    desa VARCHAR(150) NOT NULL,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    CONSTRAINT uq_dim_wilayah UNIQUE (kabupaten, desa),
    CHECK (latitude IS NULL OR latitude BETWEEN -90 AND 90),
    CHECK (longitude IS NULL OR longitude BETWEEN -180 AND 180)
);

CREATE TABLE IF NOT EXISTS dw.dim_komoditas (
    id_komoditas BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nama_komoditas VARCHAR(150) NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS dw.dim_tanggal (
    id_tanggal DATE PRIMARY KEY,
    hari INTEGER NOT NULL,
    bulan INTEGER NOT NULL,
    nama_bulan VARCHAR(20) NOT NULL,
    kuartal INTEGER NOT NULL,
    tahun INTEGER NOT NULL
);

CREATE TABLE IF NOT EXISTS dw.fact_petani_harian (
    id_snapshot BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    id_tanggal DATE NOT NULL REFERENCES dw.dim_tanggal(id_tanggal),
    kode_petani VARCHAR(50) NOT NULL,
    nama_petani VARCHAR(200),
    id_wilayah BIGINT NOT NULL REFERENCES dw.dim_wilayah(id_wilayah),
    id_komoditas BIGINT NOT NULL REFERENCES dw.dim_komoditas(id_komoditas),
    luas_lahan NUMERIC(14,2) NOT NULL DEFAULT 0,
    wilayah_sumber VARCHAR(100) NOT NULL,
    updated_at_sumber TIMESTAMPTZ,
    waktu_etl TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT uq_fact_snapshot_petani
        UNIQUE (id_tanggal, wilayah_sumber, kode_petani),
    CHECK (luas_lahan >= 0)
);

CREATE INDEX IF NOT EXISTS idx_fact_tanggal ON dw.fact_petani_harian(id_tanggal);
CREATE INDEX IF NOT EXISTS idx_fact_wilayah ON dw.fact_petani_harian(id_wilayah);
CREATE INDEX IF NOT EXISTS idx_fact_komoditas ON dw.fact_petani_harian(id_komoditas);
CREATE INDEX IF NOT EXISTS idx_fact_kode_petani ON dw.fact_petani_harian(kode_petani);
"""
conn = psycopg2.connect(DB_WAREHOUSE, connect_timeout=15)
try:
    with conn.cursor() as cur:
        cur.execute(DDL)
        # Kompatibilitas bila fact table lama sudah dibuat tanpa nama_petani.
        cur.execute("""
            ALTER TABLE dw.fact_petani_harian
            ADD COLUMN IF NOT EXISTS nama_petani VARCHAR(200);
        """)
    conn.commit()
    print("Skema dw berhasil dibuat/diperiksa.")
except Exception:
    conn.rollback()
    raise
finally:
    conn.close()


## 5. Ekstraksi data dari DB_PUSAT_SUMUT

In [ ]:
def extract_source_data():
    conn = psycopg2.connect(DB_PUSAT, connect_timeout=15)
    try:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT kode_petani, nama_petani, desa, komoditas,
                       luas_lahan, wilayah_sumber, updated_at
                FROM public.petani_pusat
                ORDER BY wilayah_sumber, kode_petani;
            """)
            rows = cur.fetchall()
        columns = ["kode_petani", "nama_petani", "desa", "komoditas",
                   "luas_lahan", "wilayah_sumber", "updated_at"]
        df = pd.DataFrame(rows, columns=columns)
        print("Jumlah data diekstrak:", len(df))
        return df
    finally:
        conn.close()

df_source = extract_source_data()
display(df_source.head())


## 6. Transformasi dan validasi
Kode petani dan wilayah sumber wajib tersedia. Nilai desa/komoditas kosong diberi label TIDAK DIKETAHUI. Luas lahan kosong menjadi nol dan nilai negatif ditolak. Duplikasi kombinasi kabupaten + kode petani tidak dibuang diam-diam.


In [ ]:
def clean_text(value, fallback=None, upper=False):
    if value is None or pd.isna(value):
        return fallback
    value = str(value).strip()
    if not value:
        return fallback
    return value.upper() if upper else value

def to_decimal(value):
    if value is None or pd.isna(value) or str(value).strip() == "":
        return Decimal("0.00")
    try:
        number = Decimal(str(value))
    except InvalidOperation:
        raise ValueError(f"Nilai luas_lahan tidak valid: {value}")
    if number < 0:
        raise ValueError(f"Luas lahan tidak boleh negatif: {value}")
    return number.quantize(Decimal("0.01"))

def transform_source_data(df):
    if df.empty:
        raise ValueError("Data sumber kosong.")
    if df[["kode_petani", "wilayah_sumber"]].isnull().any().any():
        raise ValueError("Ada kode_petani atau wilayah_sumber yang kosong.")

    result = df.copy()
    result["kode_petani"] = result["kode_petani"].astype(str).str.strip()
    result["nama_petani"] = result["nama_petani"].apply(
        lambda x: clean_text(x)
    )
    result["wilayah_sumber"] = result["wilayah_sumber"].apply(
        lambda x: clean_text(x, upper=True)
    )
    result["desa"] = result["desa"].apply(
        lambda x: clean_text(x, fallback="TIDAK DIKETAHUI")
    )
    result["komoditas"] = result["komoditas"].apply(
        lambda x: clean_text(x, fallback="TIDAK DIKETAHUI", upper=True)
    )
    result["luas_lahan"] = result["luas_lahan"].apply(to_decimal)

    duplicates = result.duplicated(
        subset=["wilayah_sumber", "kode_petani"], keep=False
    )
    if duplicates.any():
        display(result.loc[duplicates, [
            "wilayah_sumber", "kode_petani", "nama_petani", "desa"
        ]].sort_values(["wilayah_sumber", "kode_petani"]))
        raise ValueError("Duplikasi ditemukan. Periksa data sumber sebelum ETL.")

    print("Data siap dimuat:", len(result))
    print("Kabupaten:", result["wilayah_sumber"].nunique())
    print("Kombinasi kabupaten-desa:",
          result[["wilayah_sumber", "desa"]].drop_duplicates().shape[0])
    print("Komoditas:", result["komoditas"].nunique())
    print("Total luas lahan:", sum(result["luas_lahan"], Decimal("0.00")))
    return result

df_clean = transform_source_data(df_source)
display(df_clean.head())


## 7. ETL: memuat data ke DW_SUMUT
Tanggal snapshot memakai zona waktu Asia/Jakarta. ETL yang dijalankan ulang pada tanggal yang sama akan mengganti snapshot tanggal itu dalam satu transaksi. Snapshot tanggal sebelumnya tetap disimpan.


In [ ]:
SNAPSHOT_DATE = datetime.now(ZoneInfo("Asia/Jakarta")).date()
print("Tanggal snapshot:", SNAPSHOT_DATE)

def load_to_warehouse(df, snapshot_date):
    conn = psycopg2.connect(DB_WAREHOUSE, connect_timeout=15)
    try:
        with conn.cursor() as cur:
            # Dimensi tanggal
            cur.execute("""
                INSERT INTO dw.dim_tanggal
                    (id_tanggal, hari, bulan, nama_bulan, kuartal, tahun)
                VALUES (%s, %s, %s, %s, %s, %s)
                ON CONFLICT (id_tanggal) DO NOTHING;
            """, (
                snapshot_date, snapshot_date.day, snapshot_date.month,
                snapshot_date.strftime("%B"),
                (snapshot_date.month - 1) // 3 + 1,
                snapshot_date.year
            ))

            # Dimensi wilayah
            wilayah_rows = list({
                (r.wilayah_sumber, r.desa)
                for r in df.itertuples(index=False)
            })
            execute_values(cur, """
                INSERT INTO dw.dim_wilayah (kabupaten, desa)
                VALUES %s
                ON CONFLICT (kabupaten, desa) DO NOTHING;
            """, wilayah_rows)

            cur.execute("SELECT kabupaten, desa, id_wilayah FROM dw.dim_wilayah;")
            wilayah_map = {(kab, desa): wid for kab, desa, wid in cur.fetchall()}

            # Dimensi komoditas
            komoditas_rows = [(x,) for x in sorted(df["komoditas"].unique())]
            execute_values(cur, """
                INSERT INTO dw.dim_komoditas (nama_komoditas)
                VALUES %s
                ON CONFLICT (nama_komoditas) DO NOTHING;
            """, komoditas_rows)

            cur.execute("SELECT nama_komoditas, id_komoditas FROM dw.dim_komoditas;")
            komoditas_map = {nama: kid for nama, kid in cur.fetchall()}

            # Tabel fakta
            fact_rows = []
            for r in df.itertuples(index=False):
                fact_rows.append((
                    snapshot_date, r.kode_petani, r.nama_petani,
                    wilayah_map[(r.wilayah_sumber, r.desa)],
                    komoditas_map[r.komoditas], r.luas_lahan,
                    r.wilayah_sumber, r.updated_at
                ))

            # Penggantian snapshot tanggal yang sama secara atomik
            cur.execute("""
                DELETE FROM dw.fact_petani_harian
                WHERE id_tanggal = %s;
            """, (snapshot_date,))

            execute_values(cur, """
                INSERT INTO dw.fact_petani_harian (
                    id_tanggal, kode_petani, nama_petani,
                    id_wilayah, id_komoditas, luas_lahan,
                    wilayah_sumber, updated_at_sumber
                )
                VALUES %s;
            """, fact_rows, page_size=500)

        conn.commit()
        print("ETL BERHASIL")
        print("Total baris dimuat:", len(fact_rows))
        return len(fact_rows)
    except Exception:
        conn.rollback()
        print("ETL GAGAL. Transaksi di-rollback.")
        raise
    finally:
        conn.close()

loaded_count = load_to_warehouse(df_clean, SNAPSHOT_DATE)


## 8. Verifikasi data warehouse

In [ ]:
def verify_warehouse(snapshot_date):
    conn = psycopg2.connect(DB_WAREHOUSE, connect_timeout=15)
    try:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT COUNT(*), COALESCE(SUM(luas_lahan), 0)
                FROM dw.fact_petani_harian
                WHERE id_tanggal = %s;
            """, (snapshot_date,))
            print("TOTAL PETANI DAN LUAS LAHAN:", cur.fetchone())

            cur.execute("""
                SELECT wilayah_sumber, COUNT(*), SUM(luas_lahan)
                FROM dw.fact_petani_harian
                WHERE id_tanggal = %s
                GROUP BY wilayah_sumber
                ORDER BY wilayah_sumber;
            """, (snapshot_date,))
            print("\\nDATA PER KABUPATEN:")
            for row in cur.fetchall():
                print(row)

            cur.execute("""
                SELECT
                    (SELECT COUNT(*) FROM dw.dim_wilayah),
                    (SELECT COUNT(*) FROM dw.dim_komoditas),
                    (SELECT COUNT(*) FROM dw.dim_tanggal),
                    (SELECT COUNT(*) FROM dw.fact_petani_harian);
            """)
            print("\\nJUMLAH DIMENSI DAN FAKTA:", cur.fetchone())
    finally:
        conn.close()

verify_warehouse(SNAPSHOT_DATE)


## 9. Membuat empat SQL Views

In [ ]:
VIEWS_DDL = """
CREATE OR REPLACE VIEW dw.vw_petani_kabupaten AS
SELECT
    f.id_tanggal,
    f.wilayah_sumber AS kabupaten,
    COUNT(DISTINCT f.kode_petani) AS total_petani,
    SUM(f.luas_lahan) AS total_luas_lahan
FROM dw.fact_petani_harian f
GROUP BY f.id_tanggal, f.wilayah_sumber;

CREATE OR REPLACE VIEW dw.vw_petani_komoditas AS
SELECT
    f.id_tanggal,
    f.wilayah_sumber AS kabupaten,
    k.nama_komoditas,
    COUNT(DISTINCT f.kode_petani) AS total_petani,
    SUM(f.luas_lahan) AS total_luas_lahan
FROM dw.fact_petani_harian f
JOIN dw.dim_komoditas k ON f.id_komoditas = k.id_komoditas
GROUP BY f.id_tanggal, f.wilayah_sumber, k.nama_komoditas;

CREATE OR REPLACE VIEW dw.vw_petani_tren AS
SELECT
    t.id_tanggal, t.hari, t.bulan, t.nama_bulan, t.kuartal, t.tahun,
    COUNT(DISTINCT f.wilayah_sumber || '|' || f.kode_petani) AS total_petani,
    SUM(f.luas_lahan) AS total_luas_lahan
FROM dw.fact_petani_harian f
JOIN dw.dim_tanggal t ON f.id_tanggal = t.id_tanggal
GROUP BY t.id_tanggal, t.hari, t.bulan, t.nama_bulan, t.kuartal, t.tahun;

CREATE OR REPLACE VIEW dw.vw_petani_peta AS
SELECT
    f.id_tanggal, w.kabupaten, w.desa, w.latitude, w.longitude,
    COUNT(DISTINCT f.kode_petani) AS total_petani,
    SUM(f.luas_lahan) AS total_luas_lahan
FROM dw.fact_petani_harian f
JOIN dw.dim_wilayah w ON f.id_wilayah = w.id_wilayah
GROUP BY f.id_tanggal, w.kabupaten, w.desa, w.latitude, w.longitude;
"""

conn = psycopg2.connect(DB_WAREHOUSE, connect_timeout=15)
try:
    with conn.cursor() as cur:
        cur.execute(VIEWS_DDL)
    conn.commit()
    print("Empat SQL Views berhasil dibuat/diperbarui.")
except Exception:
    conn.rollback()
    raise
finally:
    conn.close()


## 10. Uji seluruh SQL Views

In [ ]:
conn = psycopg2.connect(DB_WAREHOUSE, connect_timeout=15)
try:
    with conn.cursor() as cur:
        queries = {
            "vw_petani_kabupaten": "SELECT * FROM dw.vw_petani_kabupaten ORDER BY id_tanggal DESC, kabupaten LIMIT 20;",
            "vw_petani_komoditas": "SELECT * FROM dw.vw_petani_komoditas ORDER BY id_tanggal DESC, kabupaten, nama_komoditas LIMIT 20;",
            "vw_petani_tren": "SELECT * FROM dw.vw_petani_tren ORDER BY id_tanggal;",
            "vw_petani_peta": "SELECT * FROM dw.vw_petani_peta ORDER BY id_tanggal DESC, kabupaten, desa LIMIT 20;"
        }
        for name, query in queries.items():
            cur.execute(query)
            rows = cur.fetchall()
            columns = [d[0] for d in cur.description]
            print(f"\\nVIEW {name} — {len(rows)} baris ditampilkan")
            display(pd.DataFrame(rows, columns=columns))
finally:
    conn.close()


## 11. Query ringkasan untuk dashboard

In [ ]:
conn = psycopg2.connect(DB_WAREHOUSE, connect_timeout=15)
try:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                COUNT(*) AS total_petani,
                COALESCE(SUM(luas_lahan), 0) AS total_luas_lahan,
                COUNT(DISTINCT wilayah_sumber) AS total_kabupaten,
                COUNT(DISTINCT id_komoditas) AS total_komoditas,
                MAX(id_tanggal) AS tanggal_snapshot
            FROM dw.fact_petani_harian
            WHERE id_tanggal = (
                SELECT MAX(id_tanggal) FROM dw.fact_petani_harian
            );
        """)
        print("RINGKASAN DASHBOARD:")
        print(cur.fetchone())

        cur.execute("""
            SELECT kabupaten, total_petani, total_luas_lahan
            FROM dw.vw_petani_kabupaten
            WHERE id_tanggal = (
                SELECT MAX(id_tanggal) FROM dw.fact_petani_harian
            )
            ORDER BY kabupaten;
        """)
        print("\\nPETANI PER KABUPATEN:")
        for row in cur.fetchall():
            print(row)

        cur.execute("""
            SELECT nama_komoditas, SUM(total_petani) AS total_petani
            FROM dw.vw_petani_komoditas
            WHERE id_tanggal = (
                SELECT MAX(id_tanggal) FROM dw.fact_petani_harian
            )
            GROUP BY nama_komoditas
            ORDER BY total_petani DESC;
        """)
        print("\\nPETANI PER KOMODITAS:")
        for row in cur.fetchall():
            print(row)
finally:
    conn.close()


## 12. Target hasil praktikum dan catatan
Berdasarkan dataset awal proyek:
- `DB_PUSAT_SUMUT`: 300 data.
- Deli Serdang: 100 petani.
- Karo: 100 petani.
- Langkat: 100 petani.
- Total luas lahan: 800,71 (pastikan satuan sumber adalah hektare).
- `dw.dim_wilayah`: sebelumnya 30 kombinasi kabupaten-desa.
- `dw.dim_komoditas`: sebelumnya 14 komoditas.
- `dw.fact_petani_harian`: 300 baris untuk satu tanggal snapshot.

Dimensi dapat bertambah jika kategori baru masuk. Tabel fakta akan menyimpan snapshot untuk setiap tanggal ETL. Snapshot pertama tidak otomatis menciptakan tren; tren memerlukan snapshot dari beberapa tanggal.

Koordinat desa tidak dibuat otomatis. Isi latitude/longitude hanya dari sumber geospasial yang dapat diverifikasi.

Jangan menghapus tabel warehouse secara sembarangan. Jangan membagikan connection string. Untuk dashboard publik, hindari menampilkan nama dan identitas petani.
